# init-process-group-nccl — worked example 1: Init and destroy a process group

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `init-process-group-nccl`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Every distributed worker must call `dist.init_process_group` to register its rank in the communicator and `dist.destroy_process_group` to tear it down cleanly. ARENA uses the `'nccl'` backend on multi-GPU boxes; on CPU-only runtimes the only change is the backend string `'gloo'`. The rendezvous address and port are passed through environment variables.

## Worked solution

We write a minimal worker that opens and closes a group.

1. Set the rendezvous env vars before init: `MASTER_ADDR='127.0.0.1'` and `MASTER_PORT=str(port)`. Every rank must agree on these to find each other. The `port` argument lets concurrent tests avoid collisions.
2. Call `dist.init_process_group(backend='gloo', rank=rank, world_size=world_size, timeout=...)`. A finite timeout means a missing peer fails fast instead of hanging.
3. After init, query `dist.get_rank()` and `dist.get_world_size()` and print them with a `[rank ...]` prefix so a test harness can capture them.
4. Call `dist.destroy_process_group()` before returning so the communicator is released — leaking it can wedge later runs. We demonstrate by spawning two ranks and joining them.

In [ ]:
import os
import datetime
import torch.multiprocessing as mp
import torch.distributed as dist

def worker(rank, world_size, port):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(
        backend='gloo', rank=rank, world_size=world_size,
        timeout=datetime.timedelta(seconds=20),
    )
    print(f'[rank {rank}] rank={dist.get_rank()} world={dist.get_world_size()}')
    dist.destroy_process_group()

if __name__ == '__main__':
    world_size = 2
    procs = [mp.Process(target=worker, args=(r, world_size, 29510)) for r in range(world_size)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    print('all exit codes:', [p.exitcode for p in procs])